# Reproduce — MobileNetV1 (α=0.5) / CIFAR-100
Trains the baseline from scratch with augmentation, then trains
DMC-Ultra and DMC-Board models using configs from `search.ipynb`.
Target board: ATmega2560 (Arduino Mega) — 8 KB SRAM, 256 KB Flash.


In [ ]:
import os, sys, json, torch, subprocess
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

_HERE      = os.path.dirname(os.path.abspath('.'))
_PROJ_ROOT = os.path.abspath(os.path.join('.', '../../..'))
_SHARED    = os.path.abspath(os.path.join('.', '../shared'))
sys.path.insert(0, _PROJ_ROOT)
sys.path.insert(0, _SHARED)

from development import EarlyStopper
from development.experiments.mobilenetv1 import get_model
from development.experiments.cifar100    import get_metric
from train_utils import train_compressed, evaluate_model, save_results, print_results_table


In [ ]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED        = 25
INPUT_SHAPE = (3, 32, 32)
WIDTH_MULT  = 0.5
MODELS_DIR  = 'models'
DEPLOY_DIR  = 'deployment'
DATASET_DIR = '../../../Datasets/CIFAR_100/'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DEPLOY_DIR, exist_ok=True)
torch.manual_seed(SEED)
print(f'Device: {DEVICE}')


## 1 — Augmented data loaders
CIFAR-100 augmentation is required for training from scratch.
The shared `cifar100.py` loader has no augmentation, so we define
augmented loaders here.


In [ ]:
_norm = transforms.Normalize((0.5071, 0.4867, 0.4408),
                              (0.2675, 0.2565, 0.2761))

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(), _norm,
])
test_transform = transforms.Compose([transforms.ToTensor(), _norm])

import os as _os
n_workers = min(4, _os.cpu_count())
train_dataset = datasets.CIFAR100(DATASET_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = datasets.CIFAR100(DATASET_DIR, train=False, download=True, transform=test_transform)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=n_workers, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=n_workers, pin_memory=True)

metric_fn        = get_metric()
calibration_data = next(iter(train_loader))[0].to(DEVICE)
print(f'Train: {len(train_dataset)} samples | Test: {len(test_dataset)} samples')


## 2 — Baseline MobileNetV1 (FP32)
Loads from `models/baseline.pth` if present, otherwise trains from scratch.
Training from scratch takes ~150 epochs — run on GPU.


In [ ]:
baseline_model = get_model(width_mult=WIDTH_MULT).to(DEVICE)
baseline_ckpt  = os.path.join(MODELS_DIR, 'baseline.pth')

if os.path.exists(baseline_ckpt):
    print(f'Loading baseline from {baseline_ckpt}')
    baseline_model.load_state_dict(
        torch.load(baseline_ckpt, weights_only=True)['model']
    )
else:
    print('Training baseline for 150 epochs …')
    early_stopper = EarlyStopper(
        monitor_metric='validation_loss', delta=1e-7,
        mode='min', patience=15, restore_best_state_dict=True,
    )
    criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer  = torch.optim.SGD(
        baseline_model.parameters(), lr=0.1,
        momentum=0.9, weight_decay=4e-5, nesterov=True,
    )
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=150, eta_min=1e-4,
    )
    best_acc = 0.0
    for epoch in range(150):
        baseline_model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(baseline_model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            res = baseline_model.evaluate(test_loader, {'acc': metric_fn}, device=DEVICE)
            acc = res['acc']
            print(f'  Epoch {epoch+1}: acc={acc:.2f}%')
            if acc > best_acc:
                best_acc = acc
                torch.save({'model': baseline_model.state_dict()}, baseline_ckpt)
    # Restore best checkpoint
    baseline_model.load_state_dict(torch.load(baseline_ckpt, weights_only=True)['model'])
    print(f'Baseline saved → {baseline_ckpt}  (best acc={best_acc:.2f}%)')


In [ ]:
print('Evaluating baseline …')
baseline_results = evaluate_model(
    baseline_model, test_loader, metric_fn, INPUT_SHAPE, DEVICE
)
print('Baseline:', baseline_results)


## 3 — Load compression configs from search.ipynb

In [ ]:
ultra_config = torch.load(os.path.join(MODELS_DIR, 'dmc_ultra_config.pth'),
                          weights_only=False)
board_config = torch.load(os.path.join(MODELS_DIR, 'dmc_board_config.pth'),
                          weights_only=False)
print('Configs loaded.')


## 4 — Train DMC-Ultra

In [ ]:
print('Training DMC-Ultra …')
dmc_ultra = train_compressed(
    baseline_model, ultra_config, INPUT_SHAPE,
    train_loader, test_loader, metric_fn, calibration_data,
   epochs=60, device=DEVICE, two_step=True,
)
torch.save(dmc_ultra.state_dict(), os.path.join(MODELS_DIR, 'dmc_ultra.pth'))
ultra_results = evaluate_model(dmc_ultra, test_loader, metric_fn, INPUT_SHAPE, DEVICE)
print('DMC-Ultra:', ultra_results)


## 5 — Train DMC-Board

In [ ]:
print('Training DMC-Board …')
dmc_board = train_compressed(
    baseline_model, board_config, INPUT_SHAPE,
    train_loader, test_loader, metric_fn, calibration_data,
   epochs=60, device=DEVICE, two_step=True,
)
torch.save(dmc_board.state_dict(), os.path.join(MODELS_DIR, 'dmc_board.pth'))
board_results = evaluate_model(dmc_board, test_loader, metric_fn, INPUT_SHAPE, DEVICE)
print('DMC-Board:', board_results)


## 6 — Results summary

In [ ]:
all_results = {
    'Baseline (FP32)': baseline_results,
    'DMC-Ultra':       ultra_results,
    'DMC-Board':       board_results,
}
print_results_table(all_results)
save_results(all_results, MODELS_DIR)


In [ ]:
base_size = baseline_results['size_bytes']
base_ws   = baseline_results['workspace_bytes']
for name, r in all_results.items():
    cr  = base_size / r['size_bytes']      if r['size_bytes']      else float('inf')
    wsr = base_ws   / r['workspace_bytes']  if r['workspace_bytes'] else float('inf')
    print(f"{name:<20} size_CR={cr:.1f}x  workspace_CR={wsr:.1f}x")


## 7 — Generate C headers for deployment

In [ ]:
test_input = torch.rand(INPUT_SHAPE, device=DEVICE)

for model_name, model in [('dmc_ultra', dmc_ultra), ('dmc_board', dmc_board)]:
    out_dir = os.path.join(DEPLOY_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)
    fused = model.fuse(device=DEVICE)
    fused.convert_to_c(
        INPUT_SHAPE, 'mobilenetv1_model',
        out_dir, out_dir,
        for_arduino=True,
        test_input=test_input,
    )
    print(f'C headers written to {out_dir}/')


## 8 — Compile with avr-gcc and report binary size

In [ ]:
import glob

def avr_compile(src_dir, mcu='atmega2560'):
    c_files = glob.glob(os.path.join(src_dir, '*.cpp')) + \
              glob.glob(os.path.join(src_dir, '*.c'))
    if not c_files:
        print(f'  No C/C++ files found in {src_dir}'); return
    out_elf = os.path.join(src_dir, 'model.elf')
    cmd = ['avr-g++', f'-mmcu={mcu}', '-Os', '-o', out_elf] + c_files
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('avr-g++ error:', result.stderr[:500]); return
    size_out = subprocess.run(['avr-size', out_elf],
                              capture_output=True, text=True).stdout
    print(f'\n{src_dir}:')
    print(size_out)

for model_name in ['dmc_ultra', 'dmc_board']:
    avr_compile(os.path.join(DEPLOY_DIR, model_name))
